In [1]:
import kagglehub
import pandas as pd
from pathlib import Path

# Download the dataset and locate the spreadsheet file inside the downloaded folder.
path = kagglehub.dataset_download("muratkokludataset/dry-bean-dataset")
dataset_dir = Path(path)

candidates = sorted(
    [*dataset_dir.rglob("*.xlsx"), *dataset_dir.rglob("*.xls"), *dataset_dir.rglob("*.csv")]
)

if not candidates:
    available_files = [str(file_path) for file_path in dataset_dir.rglob("*") if file_path.is_file()]
    raise FileNotFoundError(
        f"No spreadsheet files found in {dataset_dir}. Available files: {available_files}"
    )

data_file = candidates[0]
if data_file.suffix.lower() == ".csv":
    df = pd.read_csv(data_file)
else:
    df = pd.read_excel(data_file)

print("Path to dataset files:", path)
print("Loaded file:", data_file.name)
print("DataFrame shape:", df.shape)

100%|██████████| 4.54M/4.54M [00:00<00:00, 6.43MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/muratkokludataset/dry-bean-dataset/versions/1
Loaded file: Dry_Bean_Dataset.xlsx
DataFrame shape: (13611, 17)


In [2]:
df.head()

,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRation,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
0,28395,610.291,208.178117,173.888747,1.197191,0.549812,28715,190.141097,0.763923,0.988856,0.958027,0.913358,0.007332,0.003147,0.834222,0.998724,SEKER
1,28734,638.018,200.524796,182.734419,1.097356,0.411785,29172,191.272750,0.783968,0.984986,0.887034,0.953861,0.006979,0.003564,0.909851,0.998430,SEKER
2,29380,624.110,212.826130,175.931143,1.209713,0.562727,29690,193.410904,0.778113,0.989559,0.947849,0.908774,0.007244,0.003048,0.825871,0.999066,SEKER
3,30008,645.884,210.557999,182.516516,1.153638,0.498616,30724,195.467062,0.782681,0.976696,0.903936,0.928329,0.007017,0.003215,0.861794,0.994199,SEKER
4,30140,620.134,201.847882,190.279279,1.060798,0.333680,30417,195.896503,0.773098,0.990893,0.984877,0.970516,0.006697,0.003665,0.941900,0.999166,SEKER


In [3]:
df.describe()

,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRation,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4
count,13611.000000,13611.000000,13611.000000,13611.000000,13611.000000,13611.000000,13611.000000,13611.000000,13611.000000,13611.000000,13611.000000,13611.000000,13611.000000,13611.000000,13611.000000,13611.000000
mean,53048.284549,855.283459,320.141867,202.270714,1.583242,0.750895,53768.200206,253.064220,0.749733,0.987143,0.873282,0.799864,0.006564,0.001716,0.643590,0.995063
std,29324.095717,214.289696,85.694186,44.970091,0.246678,0.092002,29774.915817,59.177120,0.049086,0.004660,0.059520,0.061713,0.001128,0.000596,0.098996,0.004366
min,20420.000000,524.736000,183.601165,122.512653,1.024868,0.218951,20684.000000,161.243764,0.555315,0.919246,0.489618,0.640577,0.002778,0.000564,0.410339,0.947687
25%,36328.000000,703.523500,253.303633,175.848170,1.432307,0.715928,36714.500000,215.068003,0.718634,0.985670,0.832096,0.762469,0.005900,0.001154,0.581359,0.993703
50%,44652.000000,794.941000,296.883367,192.431733,1.551124,0.764441,45178.000000,238.438026,0.759859,0.988283,0.883157,0.801277,0.006645,0.001694,0.642044,0.996386
75%,61332.000000,977.213000,376.495012,217.031741,1.707109,0.810466,62294.000000,279.446467,0.786851,0.990013,0.916869,0.834270,0.007271,0.002170,0.696006,0.997883
max,254616.000000,1985.370000,738.860153,460.198497,2.430306,0.911423,263261.000000,569.374358,0.866195,0.994677,0.990685,0.987303,0.010451,0.003665,0.974767,0.999733


In [4]:
df.isnull().sum()

,0
Area,0
Perimeter,0
MajorAxisLength,0
MinorAxisLength,0
AspectRation,0
Eccentricity,0
ConvexArea,0
EquivDiameter,0
Extent,0
Solidity,0


In [10]:
#group the data class category
df['Class'] = df['Class'].astype('category')
print(df['Class'].value_counts())

Class
DERMASON    3546
SIRA        2636
SEKER       2027
HOROZ       1928
CALI        1630
BARBUNYA    1322
BOMBAY       522
Name: count, dtype: int64


In [5]:
# create a ML pipeline to predict the class of dry beans based on their features
from sklearn.model_selection import train_test_split
X = df.drop(columns=["Class"])
y = df["Class"]
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVC
categorical_features = X.select_dtypes(include=["object"]).columns
numerical_features = X.select_dtypes(include=["number"]).columns
categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
numerical_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numerical_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),("model",SVC())
])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



In [9]:
# Hyperparameter tuning using GridSearchCV
from sklearn.model_selection import GridSearchCV
param_grid = {
    "linear": {
        "model__C": [0.1, 1, 10],
    },
    "poly": {
        "model__C": [0.1, 1, 10],
        "model__degree": [2, 3, 4],
        "model__gamma": ["scale", 0.1, 0.01],
    },
    "rbf": {
        "model__C": [0.1, 1, 10],
        "model__gamma": ["scale", 0.1, 0.01]
    },
    "sigmoid": {
        "model__C": [0.1, 1, 10],
        "model__gamma": ["scale", 0.1, 0.01]
    },
}
results = []
fitted_models = []
conf_metrics = []
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, classification_report
for kernel in ["linear", "poly", "rbf", "sigmoid"]:
    print(f"Training SVM with {kernel} kernel...")
    pipeline.set_params(model__kernel=kernel)
    grid_search = GridSearchCV(pipeline, param_grid[kernel], cv=5, n_jobs=-1, scoring="accuracy")
    start_time = pd.Timestamp.now()
    grid_search.fit(X_train, y_train)
    elapsed_time = pd.Timestamp.now() - start_time
    best_model = grid_search.best_estimator_
    fitted_models.append(best_model)
    y_pred = best_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="weighted")
    recall = recall_score(y_test, y_pred, average="weighted")
    f1 = f1_score(y_test, y_pred, average="weighted")
    conf_matrix = confusion_matrix(y_test, y_pred)
    results.append({
        "kernel": kernel,
        "Best Parameters": grid_search.best_params_,
        "Trainig Time": elapsed_time,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "confusion_matrix": conf_matrix
    })
    print(f"Best parameters for {kernel} kernel: {grid_search.best_params_}")
    print(f"Training time for {kernel} kernel: {elapsed_time}")
    print(f"Accuracy for {kernel} kernel: {accuracy}")
    print(f"Precision for {kernel} kernel: {precision}")
    print(f"Recall for {kernel} kernel: {recall}")
    print(f"F1 Score for {kernel} kernel: {f1}")
    print(f"Confusion Matrix for {kernel} kernel:\n{conf_matrix}\n")

Training SVM with linear kernel...
Best parameters for linear kernel: {'model__C': 10}
Training time for linear kernel: 0 days 00:00:11.623468
Accuracy for linear kernel: 0.9291222915901579
Precision for linear kernel: 0.9298865681142969
Recall for linear kernel: 0.9291222915901579
F1 Score for linear kernel: 0.9293965671275564
Confusion Matrix for linear kernel:
[[243   0  15   0   0   0   3]
 [  0 117   0   0   0   0   0]
 [ 11   0 300   0   4   1   1]
 [  0   0   0 608   1   7  55]
 [  2   0   3   6 391   0   6]
 [  5   0   0   9   0 389  10]
 [  0   0   0  45   5   4 482]]

Training SVM with poly kernel...
Best parameters for poly kernel: {'model__C': 10, 'model__degree': 2, 'model__gamma': 0.1}
Training time for poly kernel: 0 days 00:04:12.016581
Accuracy for poly kernel: 0.9294895336026442
Precision for poly kernel: 0.9304373808791355
Recall for poly kernel: 0.9294895336026442
F1 Score for poly kernel: 0.9298188126326236
Confusion Matrix for poly kernel:
[[240   0  12   1   1   